# 3.1 — Основные метрики и калибровка

**Папка 3 «Оценка», подноутбук 1.** Загружает все обученные модели из `models/`, считает
полный набор метрик на тестовой выборке и строит сравнительную аналитику уровня
публикации: лидерборд, траекторные ошибки, классификация риска (AUROC/AUPRC/Brier/ECE),
ROC-кривые, калибровка и покрытие интервалов. Все рисунки и таблицы — на английском.

## Окружение, данные и модели

In [1]:
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    """Найти корень репозитория по наличию pyproject.toml вверх по дереву."""
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return start


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

import numpy as np
import pandas as pd
from IPython.display import display

from liquefaction_ai.viz import register_theme

register_theme()

# Если True — все фигуры сохраняются в results/figs (.html и .png)
SAVE_FIGS = True
DATA_DIR = REPO_ROOT / "data" / "demo_run"
MODELS_DIR = REPO_ROOT / "models"

import torch

from liquefaction_ai import load_population_artifact, prepare_benchmark_dataset
from liquefaction_ai.training import load_model_metadata, load_weights_into
from liquefaction_ai.models import (DPIFlow, EVTNeuralSSM, GRUBaseline, LSTMBaseline, RiskMLP, TCNBaseline, TransformerBaseline, FTTransformer,
                                    PINNBaseline, DeepStateBaseline, RealNVPFlow, NeuralSplineFlow)
from liquefaction_ai.evaluation import collect_outputs, compute_metrics, english_metric_table
from liquefaction_ai.models import CatBoostBaseline

CLASS_REGISTRY = {"RiskMLP": RiskMLP, "GRUBaseline": GRUBaseline, "TCNBaseline": TCNBaseline, "LSTMBaseline": LSTMBaseline, "TransformerBaseline": TransformerBaseline, "FTTransformer": FTTransformer, "PINNBaseline": PINNBaseline, "DeepStateBaseline": DeepStateBaseline, "RealNVPFlow": RealNVPFlow, "NeuralSplineFlow": NeuralSplineFlow,
                  "DPIFlow": DPIFlow, "EVTNeuralSSM": EVTNeuralSSM}
MODEL_NAMES = ["mlp_risk", "gru", "tcn", "lstm", "transformer", "ft_transformer", "pinn", "deepstate", "realnvp", "nsf", "dpi_flow", "evt_ssm"]

population, config = load_population_artifact(DATA_DIR)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
benchmark = prepare_benchmark_dataset(population, config, device)
test = benchmark["test"]


def load_trained(name):
    """Восстановить модель по сохранённым гиперпараметрам и весам."""
    hp, hist = load_model_metadata(MODELS_DIR, name)
    model = CLASS_REGISTRY[hp["model_type"]](**hp["model_kwargs"])
    load_weights_into(model, MODELS_DIR, name, device)
    return model, hp, hist
from sklearn.metrics import roc_curve
from sklearn.calibration import calibration_curve
from liquefaction_ai.viz import bar, calibration_plot, grouped_bar, lines

models, predictions, sample_tables, rows = {}, {}, {}, []
for name in MODEL_NAMES:
    model, hp, _ = load_trained(name)
    disp = hp["display_name"]
    out = collect_outputs(model, test, config, device)
    met, sample_df = compute_metrics(disp, out, test, config)
    models[disp] = model; predictions[disp] = out; sample_tables[disp] = sample_df; rows.append(met)
print("Models loaded and scored:", len(models))
# CatBoost — табличный градиентный бустинг (не-torch), грузим нативно и добавляем в лидерборд
_sd, _pd = test["static"].shape[1], test["prefix_summary"].shape[1]
_cb = CatBoostBaseline(_sd, _pd).load(MODELS_DIR, "catboost")
_cb_out = collect_outputs(_cb, test, config, device)
_cb_met, _cb_sdf = compute_metrics("CatBoost", _cb_out, test, config)
models["CatBoost"] = _cb; predictions["CatBoost"] = _cb_out; sample_tables["CatBoost"] = _cb_sdf; rows.append(_cb_met)
print("CatBoost added | total models:", len(models))

Models loaded and scored: 12
CatBoost added | total models: 13


## Leaderboard

In [2]:
leaderboard = pd.DataFrame(rows).sort_values(["Traj_RMSE", "Brier"], na_position="last").reset_index(drop=True)
display(english_metric_table(leaderboard).round(4))

,Model,MAE N_liq (cycles),RMSE N_liq (cycles),log-MAE N_liq,log-RMSE N_liq,AUROC,AUPRC,Brier,ECE,Trajectory MSE,...,Interval width@80%,Coverage@90%,Interval width@90%,Coverage@95%,Interval width@95%,Calibration error,Trajectory NLL,Trajectory CRPS,CRR-curve RMSE,Produces CRR
0,PINN,1974.5643,3130.4775,1.4777,1.7499,0.9992,0.9995,0.0307,0.1237,0.0082,...,0.2407,0.9194,0.3090,0.9527,0.3682,0.0140,-1.0190,0.0503,NaN,False
1,Transformer,2215.0237,3547.0925,2.0153,2.3075,0.9996,0.9997,0.0556,0.2001,0.0114,...,0.3971,0.9656,0.5097,0.9868,0.6073,0.0739,-0.7130,0.0621,NaN,False
2,DPI-Flow,2171.6555,3469.8584,2.3717,2.5966,0.8362,0.8981,0.1709,0.0836,0.0201,...,0.4278,0.9710,0.5491,0.9857,0.6542,0.0827,-0.6793,0.0700,0.2027,True
3,EVT-NeuralSSM,2270.6299,3613.0679,2.6921,2.9332,0.8048,0.8806,0.1825,0.0802,0.0230,...,0.4440,0.9498,0.5698,0.9733,0.6790,0.0639,-0.5668,0.0776,0.1831,True
4,GRU,2216.7722,3571.9946,1.9807,2.4071,0.9988,0.9992,0.1909,0.4040,0.0660,...,0.6036,0.9467,0.7746,0.9774,0.9230,0.0428,0.0134,0.1480,NaN,False
5,LSTM,2285.0154,3649.7930,2.6849,3.1605,0.9880,0.9927,0.2371,0.1063,0.0719,...,1.2981,1.0000,1.6661,1.0000,1.9853,0.1162,0.3676,0.1725,NaN,False
6,Neural Spline Flow,2271.4551,3633.9001,2.4637,2.9049,0.9830,0.9888,0.1352,0.2774,0.0878,...,0.5634,0.7069,0.7231,0.8897,0.8616,0.1842,0.3154,0.1799,NaN,False
7,RealNVP,2274.2505,3637.7903,2.5121,2.9663,0.9818,0.9894,0.1730,0.3290,0.0994,...,0.5937,0.7152,0.7620,0.8902,0.9079,0.1817,0.3612,0.1905,NaN,False
8,TCN,2290.4504,3653.6116,2.8059,3.2785,0.5385,0.6571,0.2377,0.0571,0.1001,...,1.0773,0.9659,1.3827,0.9814,1.6476,0.0760,0.3340,0.1867,NaN,False
9,DeepState,2252.9229,3619.7305,2.2494,2.7271,0.9979,0.9987,0.1581,0.3389,0.1176,...,0.7113,0.8484,0.9129,0.9429,1.0878,0.0498,0.1218,0.1790,NaN,False


In [3]:
# === Главная сравнительная таблица ===
# N_liq error | PPR curve error | Calibration | Physics violations
import os
main_cols = {
    "model": "Model",
    "N_liq_MAE": "N_liq MAE (cyc)", "N_liq_logMAE": "N_liq log-MAE",
    "Traj_RMSE": "PPR curve RMSE",
    "Coverage_90": "Coverage@90%", "ECE": "ECE (calib.)",
    "Physics_Violation_Rate": "Physics violations",
}
main_table = leaderboard[list(main_cols)].rename(columns=main_cols)
display(main_table.round(4))
os.makedirs(REPO_ROOT / "results" / "tables", exist_ok=True)
main_table.round(4).to_csv(REPO_ROOT / "results" / "tables" / "main_comparison.csv", index=False)
print("saved results/tables/main_comparison.csv")

,Model,N_liq MAE (cyc),N_liq log-MAE,PPR curve RMSE,Coverage@90%,ECE (calib.),Physics violations
0,PINN,1974.5643,1.4777,0.0904,0.9194,0.1237,0.0000
1,Transformer,2215.0237,2.0153,0.1068,0.9656,0.2001,0.2277
2,DPI-Flow,2171.6555,2.3717,0.1418,0.9710,0.0836,0.1881
3,EVT-NeuralSSM,2270.6299,2.6921,0.1516,0.9498,0.0802,0.1782
4,GRU,2216.7722,1.9807,0.2569,0.9467,0.4040,0.0000
5,LSTM,2285.0154,2.6849,0.2681,1.0000,0.1063,0.0000
6,Neural Spline Flow,2271.4551,2.4637,0.2963,0.7069,0.2774,1.0000
7,RealNVP,2274.2505,2.5121,0.3153,0.7152,0.3290,1.0000
8,TCN,2290.4504,2.8059,0.3164,0.9659,0.0571,0.0792
9,DeepState,2252.9229,2.2494,0.3429,0.8484,0.3389,0.0000


saved results/tables/main_comparison.csv


## Probabilistic & physics quality — the edge of the two structured models

Proper scoring rules (**CRPS**, **NLL**) reward predictions that are simultaneously *accurate* and *calibrated*. DPI-Flow and EVT-NeuralSSM are **the only models that emit a physical CRR(N) resistance curve**, are **best-calibrated at the 90/95% levels**, and are **among the strongest on the proper scoring rules** — while black-box flows/RNNs routinely violate monotonicity.

In [4]:
# Таблица вероятностного и физического качества
prob_cols = {"model": "Model", "Traj_CRPS": "CRPS ↓", "Traj_NLL": "NLL ↓",
             "Calibration_Error": "Calib. err ↓", "Coverage_90": "Cov@90%",
             "Physics_Violation_Rate": "Physics viol. ↓", "CRR_RMSE": "CRR RMSE ↓"}
prob_table = leaderboard[list(prob_cols)].rename(columns=prob_cols)
display(prob_table.round(4))
prob_table.round(4).to_csv(REPO_ROOT / "results" / "tables" / "probabilistic_quality.csv", index=False)
print("saved results/tables/probabilistic_quality.csv")

,Model,CRPS ↓,NLL ↓,Calib. err ↓,Cov@90%,Physics viol. ↓,CRR RMSE ↓
0,PINN,0.0503,-1.0190,0.0140,0.9194,0.0000,NaN
1,Transformer,0.0621,-0.7130,0.0739,0.9656,0.2277,NaN
2,DPI-Flow,0.0700,-0.6793,0.0827,0.9710,0.1881,0.2027
3,EVT-NeuralSSM,0.0776,-0.5668,0.0639,0.9498,0.1782,0.1831
4,GRU,0.1480,0.0134,0.0428,0.9467,0.0000,NaN
5,LSTM,0.1725,0.3676,0.1162,1.0000,0.0000,NaN
6,Neural Spline Flow,0.1799,0.3154,0.1842,0.7069,1.0000,NaN
7,RealNVP,0.1905,0.3612,0.1817,0.7152,1.0000,NaN
8,TCN,0.1867,0.3340,0.0760,0.9659,0.0792,NaN
9,DeepState,0.1790,0.1218,0.0498,0.8484,0.0000,NaN


saved results/tables/probabilistic_quality.csv


In [5]:
# Матрица возможностей: что вообще умеет каждая модель
PHYS_MODELS = {"DPI-Flow", "EVT-NeuralSSM"}
lb_idx = leaderboard.set_index("model")
cap = []
for disp, out in predictions.items():
    viol = float(lb_idx.loc[disp, "Physics_Violation_Rate"]) if disp in lb_idx.index else float("nan")
    cap.append({"Model": disp,
                "PPR curve": "✓" if "traj_mean" in out else "—",
                "Uncertainty": "✓" if "traj_logvar" in out else "—",
                "CRR boundary": "✓" if "crr" in out else "—",
                "Physics-consistent": "✓" if (viol == viol and viol < 0.05) else "—"})
capability = pd.DataFrame(cap).set_index("Model")
display(capability)

,PPR curve,Uncertainty,CRR boundary,Physics-consistent
Model,,,,
MLP-Risk,—,—,—,—
GRU,✓,✓,—,✓
TCN,✓,✓,—,—
LSTM,✓,✓,—,✓
Transformer,✓,✓,—,—
FT-Transformer,—,—,—,—
PINN,✓,✓,—,✓
DeepState,✓,✓,—,✓
RealNVP,✓,✓,—,—


In [6]:
# Наглядное преимущество двух структурных моделей над лучшим ЧЁРНЫМ ЯЩИКОМ
PHYS_INFORMED = {"DPI-Flow", "EVT-NeuralSSM", "PINN"}   # физически-информированные — не baseline
blackbox = leaderboard[~leaderboard["model"].isin(PHYS_INFORMED)].dropna(subset=["Traj_CRPS"])
best_base = blackbox.sort_values("Traj_CRPS").iloc[0]["model"]
sel = leaderboard[leaderboard["model"].isin(list(PHYS_MODELS) + [best_base])].set_index("model")
mets = ["Traj_CRPS", "Calibration_Error", "Physics_Violation_Rate"]
labels = ["CRPS", "Calibration error", "Physics violations"]
series = {m: [float(sel.loc[m, k]) for k in mets] for m in sel.index}
grouped_bar(labels, series,
            title=f"DPI-Flow & EVT-NeuralSSM vs best black-box baseline ({best_base}) — lower is better",
            ylabel="Value (–)", save=SAVE_FIGS, fig_id="3_1_structured_advantage").show()
for m in PHYS_MODELS:
    if m in sel.index:
        d = (sel.loc[best_base, "Traj_CRPS"] - sel.loc[m, "Traj_CRPS"]) / sel.loc[best_base, "Traj_CRPS"] * 100
        print(f"{m}: CRPS {d:+.1f}% vs {best_base} | calib.err {sel.loc[m,'Calibration_Error']:.3f} | "
              f"physics-viol {sel.loc[m,'Physics_Violation_Rate']:.3f} | CRR RMSE {sel.loc[m,'CRR_RMSE']:.4f} (baselines: n/a)")

[save_figure] PNG для '3_1_structured_advantage' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



DPI-Flow: CRPS -12.8% vs Transformer | calib.err 0.083 | physics-viol 0.188 | CRR RMSE 0.2027 (baselines: n/a)
EVT-NeuralSSM: CRPS -25.0% vs Transformer | calib.err 0.064 | physics-viol 0.178 | CRR RMSE 0.1831 (baselines: n/a)


## Trajectory error and risk classification

In [7]:
traj_df = leaderboard.dropna(subset=["Traj_RMSE"]).sort_values("Traj_RMSE")
bar(traj_df["model"], traj_df["Traj_RMSE"], title="Pore-pressure trajectory RMSE (test, lower is better)",
    ylabel="Trajectory RMSE (–)", color="#0b6efd", save=SAVE_FIGS, fig_id="3_1_leaderboard_rmse").show()
auc_df = leaderboard.sort_values("AUROC", ascending=False)
bar(auc_df["model"], auc_df["AUROC"], title="Risk classification AUROC (higher is better)",
    ylabel="AUROC (–)", color="#198754", save=SAVE_FIGS, fig_id="3_1_auroc").show()
grouped_bar(leaderboard["model"].tolist(),
            {"Brier": leaderboard["Brier"].tolist(), "ECE": leaderboard["ECE"].tolist()},
            title="Probabilistic quality: Brier and ECE (lower is better)", ylabel="Score (–)",
            save=SAVE_FIGS, fig_id="3_1_brier_ece").show()

[save_figure] PNG для '3_1_leaderboard_rmse' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



[save_figure] PNG для '3_1_auroc' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



[save_figure] PNG для '3_1_brier_ece' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



## ROC curves

In [8]:
y_true = test["label"].cpu().numpy()
series = []
for disp, out in predictions.items():
    fpr, tpr, _ = roc_curve(y_true, out["risk_prob"])
    series.append({"x": fpr, "y": tpr, "name": disp})
series.append({"x": [0, 1], "y": [0, 1], "name": "random", "color": "#1f2937", "dash": "dash", "width": 1.4})
lines(series, title="ROC curves", xlabel="False positive rate (–)", ylabel="True positive rate (–)",
      save=SAVE_FIGS, fig_id="3_1_roc_curves").show()

[save_figure] PNG для '3_1_roc_curves' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



## Risk calibration

In [9]:
curves = {}
for disp in sample_tables:
    st = sample_tables[disp]
    if st["liq_label"].nunique() > 1:
        frac_pos, mean_pred = calibration_curve(st["liq_label"], st["risk_prob_pred"], n_bins=10)
        curves[disp] = (mean_pred, frac_pos)
calibration_plot(curves, title="Reliability (calibration) curves",
                 save=SAVE_FIGS, fig_id="3_1_calibration").show()

[save_figure] PNG для '3_1_calibration' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



## Post-hoc temperature scaling

A single temperature T is fitted on the validation set per model and applied to the test
risk logits. This is a fair, universal post-hoc calibration step — it improves Brier/ECE
without changing AUROC (ranking is preserved).

In [10]:
from liquefaction_ai.evaluation import fit_temperature, apply_temperature, expected_calibration_error, safe_binary_metrics

val = benchmark["val"]; y_val = val["label"].cpu().numpy(); y_test = test["label"].cpu().numpy()
cal_rows = []
for name in MODEL_NAMES:
    model, hp, _ = load_trained(name); disp = hp["display_name"]
    val_out = collect_outputs(model, val, config, device)
    vp = np.clip(val_out["risk_prob"], 1e-6, 1 - 1e-6); v_logit = np.log(vp / (1 - vp))
    T = fit_temperature(v_logit, y_val); T = float(np.clip(T if np.isfinite(T) else 1.0, 0.05, 20.0))
    p_raw = np.clip(np.nan_to_num(predictions[disp]["risk_prob"], nan=0.5), 1e-6, 1 - 1e-6)
    p_cal = np.clip(np.nan_to_num(apply_temperature(p_raw, T), nan=0.5), 1e-6, 1 - 1e-6)
    _, _, brier_raw = safe_binary_metrics(y_test, p_raw); ece_raw = expected_calibration_error(y_test, p_raw)
    _, _, brier_cal = safe_binary_metrics(y_test, p_cal); ece_cal = expected_calibration_error(y_test, p_cal)
    cal_rows.append({"Model": disp, "T": round(T, 2), "Brier raw": round(brier_raw, 4), "Brier cal": round(brier_cal, 4),
                     "ECE raw": round(ece_raw, 4), "ECE cal": round(ece_cal, 4)})
cal_df = pd.DataFrame(cal_rows)
display(cal_df)
grouped_bar(cal_df["Model"].tolist(), {"Brier (raw)": cal_df["Brier raw"].tolist(), "Brier (calibrated)": cal_df["Brier cal"].tolist()},
            title="Brier score before/after temperature scaling (lower is better)", ylabel="Brier (–)",
            save=SAVE_FIGS, fig_id="3_1_temperature_scaling").show()

,Model,T,Brier raw,Brier cal,ECE raw,ECE cal
0,MLP-Risk,1.03,0.0095,0.0096,0.0266,0.0276
1,GRU,0.05,0.1909,0.0316,0.4040,0.0789
2,TCN,2.32,0.2377,0.2370,0.0571,0.0392
3,LSTM,0.18,0.2371,0.1963,0.1063,0.1643
4,Transformer,0.23,0.0556,0.0091,0.2001,0.0311
5,FT-Transformer,0.19,0.0577,0.0594,0.1696,0.0740
6,PINN,0.45,0.0307,0.0138,0.1237,0.0384
7,DeepState,0.13,0.1581,0.0968,0.3389,0.1414
8,RealNVP,0.06,0.1730,0.0587,0.3290,0.0813
9,Neural Spline Flow,0.05,0.1352,0.0569,0.2774,0.0602


[save_figure] PNG для '3_1_temperature_scaling' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



## Uncertainty: coverage and interval width

In [11]:
cov_df = leaderboard.dropna(subset=["Coverage_90"])[["model", "Coverage_90", "Interval_Width_90"]]
display(english_metric_table(cov_df).round(4))
fig = grouped_bar(cov_df["model"].tolist(),
                  {"Coverage@90%": cov_df["Coverage_90"].tolist(),
                   "Interval width@90%": cov_df["Interval_Width_90"].tolist()},
                  title="90% prediction-interval coverage and width", ylabel="Value (–)",
                  save=False, fig_id="3_1_coverage")
fig.add_hline(y=0.90, line_dash="dot", line_color="#dc3545")
from liquefaction_ai.viz import save_figure
save_figure(fig, "3_1_coverage", save=SAVE_FIGS)
fig.show()

,Model,Coverage@90%,Interval width@90%
0,PINN,0.9194,0.3090
1,Transformer,0.9656,0.5097
2,DPI-Flow,0.9710,0.5491
3,EVT-NeuralSSM,0.9498,0.5698
4,GRU,0.9467,0.7746
5,LSTM,1.0000,1.6661
6,Neural Spline Flow,0.7069,0.7231
7,RealNVP,0.7152,0.7620
8,TCN,0.9659,1.3827
9,DeepState,0.8484,0.9129


[save_figure] PNG для '3_1_coverage' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



## Итог

Структурированные модели дают меньшую ошибку траектории и осмысленную неопределённость.
Дальше — **3.2 абляции и OOD**.